# Dalhousie university
## CPI index construction following  National Administrative Statistic Department - DANE of Colombia
#### Objective: Explore actual data to replicate the DANE methodology of CPI construction.

## 1. Loading and preparation of Data

### 1.1. Environment preparation

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt

# options for data display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


### 1.2. Data loading

In [ ]:
# load data using prjoect path
data = pd.read_csv(wd_d + "A1_clean.csv")

In [ ]:
# shows last ten (10) registers
data.tail(10)

### 1.3 Data preprocessing
We create new columns. Also split and clean data to product databases. 

In [ ]:
# erase null products
data = data[~data["precio"].isna()]

In [ ]:
# set date as index
data = data.set_index(data["fecha"])

In [ ]:
# drop duplicated data
data = data.drop_duplicates(["Descripcion", "fecha"])

In [ ]:
# create year, month, date columns
data['fecha'] = data['fecha'].astype(str)
data[["anio", "mes", "dia"]] = data["fecha"].str.split(pat = "-", expand = True)

In [ ]:
# change column type and weekday column
data["fecha"] = pd.to_datetime(data["fecha"])
data["weekday"] = data["fecha"].dt.day_name() # it would be important for analyzing weekend discounts and other seasonal discounts.

#### 1.3.1 Beer

In [ ]:
# filter data for only beer 
cervezadf = data[data["palabra"]=="cerveza"]

In [ ]:
# show last 10 register of beer data (data is dirty, it appear products like wine. Needs cleaning)
cervezadf.tail(10)

In [ ]:
# count unique varieties where liquid unit is null
cervezadf[cervezadf["unidad_liquida"].isna()].Descripcion.value_counts()

In [ ]:
# following the previous step, we erase that varieties
cervezadf = cervezadf[~cervezadf["unidad_liquida"].isna()]

In [ ]:
# looks for varieties that not has (cerv or pola or paulaner) in the description.
cervezadf[~cervezadf["Descripcion"].str.contains("cerv|pola|paulaner")].Descripcion.value_counts()

In [ ]:
# applied last filter followin printed varieties
cervezadf = cervezadf[cervezadf["Descripcion"].str.contains("cerv|pola|paulaner")]

In [ ]:
# look of actual data varieties
cervezadf.Descripcion.value_counts()

In [ ]:
# there are some noise yet, print this varieties
cervezadf[cervezadf["Descripcion"].str.contains("vaso|termo|pack ron")].Descripcion.value_counts()

In [ ]:
# applied last filter
cervezadf = cervezadf[~cervezadf["Descripcion"].str.contains("vaso|termo|pack ron")]

In [ ]:
# look of actual data varieties again
cervezadf.Descripcion.value_counts()
# data looks clear

In [ ]:
# print data 
cervezadf.tail(10)

#### 1.3.2 Cigarettes

In [ ]:
# filter data for only cigarettes
cigarrillosdf = data[data["palabra"] == "cigarrillos"]

In [ ]:
# varieties in the actual data
cigarrillosdf.Descripcion.value_counts()

In [ ]:
# filter data when (cig or cgllo) are present in description
cigarrillosdf = cigarrillosdf[cigarrillosdf["Descripcion"].str.contains("cig|cgllo")]

In [ ]:
# count actual unique varieties
cigarrillosdf.Descripcion.value_counts()

In [ ]:
# following the previous step, we erase one noise variety
cigarrillosdf = cigarrillosdf[~cigarrillosdf["Descripcion"].str.contains("adaptador")]

## 2. Data analysis

We want to take a look of the data.

### 2.1. Beer

In [ ]:
# First a table with descriptive statistics of the interes variable
stats_desc = cervezadf['precio'].describe()

stats_desc = stats_desc.apply(lambda x: f'COP ${x:,.1f}')

print(stats_desc)

In [ ]:
# poker and aguila are most selled varieties, filter data for this brands
bidf = cervezadf[cervezadf["Descripcion"].str.contains("aguila|poker")]

In [ ]:
bidf.Descripcion.value_counts()

In [ ]:
plotdf = bidf[["fecha", "Descripcion", "precio"]]

# Pivot the DataFrame to a wide format
plotdf = plotdf.pivot(index='fecha', columns='Descripcion', values='precio')

# Resetting the index to have a flat DataFrame (optional)
plotdf = plotdf.reset_index()

# Combina las columnas de año, mes y día en una columna de tipo datetime
plotdf['fecha'] = pd.to_datetime(plotdf["fecha"])

# Plot each product as a time series
plt.figure(figsize=(15, 6))

for column in plotdf.columns[1:]:  # Skip the 'Date' column
    plt.plot(plotdf['fecha'], plotdf[column], label=column,  marker='o', linestyle='--', linewidth=1, markersize=2)

# Add labels, title, and legend
plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Nominal prices')
plt.legend(title='Variety', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
# histogram
plt.figure(figsize=(12, 8))
sns.boxplot(data=bidf, x='Descripcion', y='precio')
plt.xticks(rotation=90)
plt.title('Price distribution per product')
plt.show()

In [ ]:
# Price histogram
plt.figure(figsize=(12, 8))
sns.histplot(data=bidf, x='precio', hue='Descripcion', kde=True, bins=30)
plt.title('Total price distribution per product')
plt.show()

# Coeficient variation table
bidf_cv = bidf.groupby('Descripcion').agg(
    Mean=('precio', 'mean'),
    SD=('precio', 'std')
)
bidf_cv['CV'] = bidf_cv['SD'] / bidf_cv['Mean']
bidf_cv

In [ ]:
# initial date and final date

# grouping per prouct and calculate initial and final date
dates_horizon = bidf.groupby('Descripcion')['fecha'].agg(initial_date='min', final_date='max')

dates_horizon['total_days'] = (dates_horizon['final_date'] - dates_horizon['initial_date']).dt.days

# show table
dates_horizon

Plot similar products

In [ ]:
plotdf = bidf[["fecha", "Descripcion", "precio"]]

plotdf = plotdf[plotdf["Descripcion"].str.contains("cerv. lata pack 6")]

# Pivot the DataFrame to a wide format
plotdf = plotdf.pivot(index='fecha', columns='Descripcion', values='precio')

# Resetting the index to have a flat DataFrame (optional)
plotdf = plotdf.reset_index()

# Combina las columnas de año, mes y día en una columna de tipo datetime
plotdf['fecha'] = pd.to_datetime(plotdf["fecha"])

# Plot each product as a time series
plt.figure(figsize=(15, 6))

for column in plotdf.columns[1:]:  # Skip the 'Date' column
    plt.plot(plotdf['fecha'], plotdf[column], label=column)

# Add labels, title, and legend
plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Nominal prices')
plt.legend(title='Variety', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Show the plot
plt.show()

## 3. CPI construction

### 3.1 Beer
With this methodology we want to replicate the Administrative Statistic Deparment of Colombia - DANE. They apply a mixed index approximation. With this they intend to incorporate a flexible part of the index which will be able to capture variation in consumption trends. 

We describe the steps following:

#### 3.1.1. Simple relatives per variety:
Defined as $p_{i,t}/p_{i,t-1}$. Where $p_t$ is the price of the variety i at moment t. A variety is defined as a good that has some explicit characteristics that can be fixed over time. Example: Dry rice sold at the Atlantic Superstore on Quinpool Street, no name brand, packaged in a plastic bag for 500 grams. 

This variety is selected to represent the elementary product classes. It is selected according to the rule: the best-selling variety.

In [ ]:
# Selecting best-selling varietys 
cervezadf.Descripcion.value_counts()

In [ ]:
# poker and aguila are most selled varieties, filter data for this brands
bidf = cervezadf[cervezadf["Descripcion"].str.contains("aguila|poker")]

In [ ]:
# calculate simple relatives
bsrdf = pd.DataFrame()
for producto in bidf["Descripcion"].unique():
    temp = bidf[bidf["Descripcion"] == producto].reset_index(drop = True)
    temp = temp.sort_values(["fecha"])
    temp["Relativo"] = np.nan
    c = 1
    lista = temp["fecha"].unique()
    for dia in lista:
        if c > 1:
            actual_value = float(temp[temp["fecha"]==dia]["precio"].iloc[0])
            actual_day = dia
            if (((actual_day-past_day)/ np.timedelta64(1, 'D')) == 1).bool():
                relativo = actual_value / past_value
                temp.loc[temp["fecha"]==dia, "Relativo"] = relativo 
            else:
                relativo = np.nan
                temp.loc[temp["fecha"]==dia, "Relativo"] = relativo 
        past_day = temp[temp["fecha"]==dia]["fecha"]
        past_value = float(temp[temp["fecha"]==dia]["precio"].iloc[0])
        c+=1
    bsrdf = pd.concat([bsrdf, temp])

In [ ]:
bsrdf.head(10)

#### 3.1.2. Geometric average of simple relative per product classes
It works at the lowest level of aggregation. Implies a geometric mean that varies less to extreme lowest and greates values, it is defined as: 
$$\sqrt[n]{\prod_{i=1}^{n}{s_{i, t}}}$$
Where $s_{i,t}$ represents the simple relative of the variety i at moment t. 


In [ ]:
# function to ignor na geometric mean
def geometric_mean_ignore_nan(row):
    return gmean(row.dropna())  # Drop NaNs and calculate geometric mean

# calculates the geometric mean by product classes
bagmdf = bsrdf.groupby(["fecha"])['Relativo'].apply(geometric_mean_ignore_nan).reset_index()
bagmdf["Descripcion"] = "All"
# calculates the geometric mean by varieties (same relative ?)
bvgmdf = bsrdf.groupby(["fecha", "Descripcion"])['Relativo'].apply(gmean).reset_index()

# save all into a dataframe
bgmdf = pd.concat([bagmdf, bvgmdf], ignore_index=True)

# Reorder and rename columns
bgmdf = bgmdf[['fecha', 'Descripcion', 'Relativo']]  # Reorder columns
bgmdf.columns = ['fecha', 'Descripcion', 'gm']  # Rename columns


#### 3.1.3 Plot CPI

In [ ]:
# Pivot the DataFrame to a wide format
bgmdfw = bgmdf.pivot(index='fecha', columns='Descripcion', values='gm')

# Resetting the index to have a flat DataFrame (optional)
bgmdfw = bgmdfw.reset_index()

In [ ]:
# Combina las columnas de año, mes y día en una columna de tipo datetime
bgmdfw['fecha'] = pd.to_datetime(bgmdfw["fecha"])

# Plot each product as a time series
plt.figure(figsize=(15, 6))

for column in bgmdfw.columns[1:]:  # Skip the 'Date' column
    plt.plot(bgmdfw['fecha'], bgmdfw[column], label=column, marker='o', linestyle='--', linewidth=1, markersize=2)

# Add labels, title, and legend
plt.xlabel('Date')
plt.ylabel('Price change')
plt.title('Daily inflation')
plt.legend(title='Variety', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Show the plot
plt.show()

In [ ]:
bgmdfw.head(20)

In [ ]:
# Plot each product as a time series
plt.figure(figsize=(15, 6))

plt.plot(bgmdfw['fecha'], bgmdfw[bgmdfw.columns[1]], label=bgmdfw.columns[1], marker='o', linestyle='-', linewidth=2, markersize=2)
for column in bgmdfw.columns[[15, 18, 21]]:  # Skip the 'Date' column
    plt.plot(bgmdfw['fecha'], bgmdfw[column], label=column, marker='o', linestyle='--', linewidth=1, markersize=2)

# Add labels, title, and legend
plt.xlabel('Date')
plt.ylabel('Price change')
plt.title('Daily inflation')
plt.legend(title='Variety', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)

# Show the plot
plt.show()